# Avance Fase 3 — Semana 2

## Núcleo algorítmico, eficiencia y programación orientada a objetos

**Proyecto:** Análisis reproducible de fallas en maquinaria industrial con AI4I 2020  
**Asignatura:** Programación para la Ciencia de Datos  
**Grupo:** 9  
**Estudiante:** Alexander Sepúlveda Ormazábal  

## Introducción

Durante las fases anteriores se definió la problemática del mantenimiento predictivo y se construyó un pipeline reproducible para preparar el conjunto de datos AI4I 2020. Este procesamiento permitió seleccionar las variables analíticas, excluir identificadores y variables que producirían fuga de información, codificar el tipo de producto, escalar las variables operacionales y generar conjuntos separados de entrenamiento y prueba.

En la Fase 3 se reorganiza ese procesamiento mediante programación orientada a objetos. El propósito es conservar la lógica y los resultados de la Fase 2, pero distribuir las responsabilidades en clases y módulos que puedan probarse, reutilizarse y modificarse de manera independiente.

Además, se compararán dos implementaciones de una misma operación para evaluar su tiempo de ejecución y consumo de memoria. Finalmente, se comprobará que la arquitectura orientada a objetos produzca los mismos conjuntos procesados que la implementación de la Fase 2.

## Objetivo general

Reorganizar el pipeline de preprocesamiento de AI4I 2020 mediante programación estructurada y orientada a objetos, conservando los resultados de la Fase 2 y mejorando la modularidad, reutilización, trazabilidad y capacidad de prueba del código.

## Objetivos específicos

1. Identificar las responsabilidades que forman el pipeline de la Fase 2.
2. Distribuir esas responsabilidades en funciones y clases claramente delimitadas.
3. Aplicar encapsulamiento, herencia y polimorfismo en los componentes que lo justifiquen.
4. Comparar dos implementaciones equivalentes mediante mediciones de tiempo y memoria.
5. Determinar si la recursividad aporta una solución adecuada al proyecto o si resulta preferible utilizar división funcional.
6. Seleccionar un patrón de diseño solo si permite resolver un problema concreto de la arquitectura.
7. Probar el funcionamiento del código mediante casos normales, límite y de excepción.
8. Comprobar que los resultados de la Fase 3 coincidan con los archivos procesados en la Fase 2. 

## 1. Verificación del entorno reproducible

Antes de reorganizar el pipeline se verifica que el notebook esté utilizando el entorno virtual del proyecto. También se registran las versiones de Python y de las principales librerías utilizadas.

Esta comprobación permite detectar diferencias de entorno antes de ejecutar el procesamiento. La semilla se mantiene en 42 para reproducir la misma separación de entrenamiento y prueba utilizada en la Fase 2.

In [1]:
import sys
import platform
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import sklearn

SEMILLA = 42
np.random.seed(SEMILLA)

interprete = Path(sys.executable).resolve()

if ".venv" not in interprete.parts:
    raise RuntimeError(
        "El notebook no está utilizando el entorno virtual .venv del proyecto."
    )

print("Fase 3 · Grupo 9 ·", date.today().isoformat())
print("Intérprete:", interprete)
print("Carpeta de trabajo:", Path.cwd().resolve())
print("Sistema:", platform.platform())
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Semilla:", SEMILLA)


Fase 3 · Grupo 9 · 2026-10-04
Intérprete: C:\proyecto-grupo9-mcdi500\.venv\Scripts\python.exe
Carpeta de trabajo: C:\proyecto-grupo9-mcdi500\F3
Sistema: Windows-11-10.0.26200-SP0
Python: 3.13.0
NumPy: 2.5.3
pandas: 3.0.5
scikit-learn: 1.9.1
Semilla: 42


## 2. Localización del proyecto y definición de rutas

Las rutas del proyecto se construyen a partir de la raíz del repositorio. De esta forma, el notebook no depende de una dirección personal ni de una ubicación específica del computador.

La raíz se reconoce mediante la presencia de `README.md` y de las carpetas `F1` y `F2`. Después se comprueba la existencia del archivo original y de los conjuntos procesados en la Fase 2, ya que estos últimos se utilizarán como referencia para validar la nueva implementación.

In [2]:
def buscar_raiz(inicio):
    """Busca la raíz del repositorio desde una carpeta inicial."""
    actual = Path(inicio).resolve()

    for candidata in [actual, *actual.parents]:
        tiene_readme = (candidata / "README.md").is_file()
        tiene_f1 = (candidata / "F1").is_dir()
        tiene_f2 = (candidata / "F2").is_dir()

        if tiene_readme and tiene_f1 and tiene_f2:
            return candidata

    raise FileNotFoundError(
        "No se encontró la raíz del repositorio."
    )


RAIZ = buscar_raiz(Path.cwd())

ARCHIVO_CRUDO = (
    RAIZ / "F1" / "data" / "raw" / "ai4i2020.csv"
)

ARCHIVO_ENTRENAMIENTO_F2 = (
    RAIZ
    / "F2"
    / "data"
    / "processed"
    / "ai4i2020_entrenamiento_procesado.csv"
)

ARCHIVO_PRUEBA_F2 = (
    RAIZ
    / "F2"
    / "data"
    / "processed"
    / "ai4i2020_prueba_procesado.csv"
)

DIR_PROCESADO_F3 = RAIZ / "F3" / "data" / "processed"
DIR_DOCS_F3 = RAIZ / "F3" / "docs"
DIR_SRC_F3 = RAIZ / "F3" / "src"

# Estas carpetas se crean solamente si todavía no existen.
for carpeta in [DIR_PROCESADO_F3, DIR_DOCS_F3, DIR_SRC_F3]:
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVOS_ENTRADA = {
    "Datos originales": ARCHIVO_CRUDO,
    "Entrenamiento F2": ARCHIVO_ENTRENAMIENTO_F2,
    "Prueba F2": ARCHIVO_PRUEBA_F2,
}

for nombre, ruta in ARCHIVOS_ENTRADA.items():
    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {nombre}: {ruta}"
        )

print("Raíz del repositorio:", RAIZ)

for nombre, ruta in ARCHIVOS_ENTRADA.items():
    print(f"{nombre}: {ruta.relative_to(RAIZ)}")

print(
    "Salida de F3:",
    DIR_PROCESADO_F3.relative_to(RAIZ),
)


Raíz del repositorio: C:\proyecto-grupo9-mcdi500
Datos originales: F1\data\raw\ai4i2020.csv
Entrenamiento F2: F2\data\processed\ai4i2020_entrenamiento_procesado.csv
Prueba F2: F2\data\processed\ai4i2020_prueba_procesado.csv
Salida de F3: F3\data\processed


## 3. Carga de datos y resultados de referencia

La implementación orientada a objetos debe comenzar desde el archivo original de AI4I 2020 y reproducir el procesamiento realizado en la Fase 2.

Los conjuntos procesados de entrenamiento y prueba se cargan como resultados de referencia. Estos archivos funcionarán como un control para comprobar posteriormente que la reorganización del código no cambió las filas, las columnas ni los valores obtenidos anteriormente.

La carga se encapsula inicialmente en una función con una responsabilidad única: verificar y leer un archivo CSV.

In [3]:
def cargar_csv(ruta, nombre):
    """Comprueba y carga un archivo CSV."""
    ruta = Path(ruta)

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {nombre}: {ruta}"
        )

    try:
        datos = pd.read_csv(
            ruta,
            encoding="utf-8-sig",
        )
    except pd.errors.EmptyDataError as error:
        raise ValueError(
            f"El archivo {nombre} está vacío."
        ) from error

    if datos.empty:
        raise ValueError(
            f"El archivo {nombre} no contiene registros."
        )

    print(
        f"{nombre}: "
        f"{datos.shape[0]} filas x "
        f"{datos.shape[1]} columnas"
    )

    return datos


df_crudo = cargar_csv(
    ARCHIVO_CRUDO,
    "datos originales",
)

entrenamiento_f2 = cargar_csv(
    ARCHIVO_ENTRENAMIENTO_F2,
    "entrenamiento de F2",
)

prueba_f2 = cargar_csv(
    ARCHIVO_PRUEBA_F2,
    "prueba de F2",
)

datos originales: 10000 filas x 14 columnas
entrenamiento de F2: 8000 filas x 9 columnas
prueba de F2: 2000 filas x 9 columnas


### Verificación de dimensiones

Se comprueba que el conjunto original y los archivos procesados tengan las mismas dimensiones registradas en la Fase 2. Esta validación permite detectar archivos incompletos o diferentes antes de construir el nuevo pipeline.

In [4]:
DIMENSIONES_ESPERADAS = {
    "datos originales": (10000, 14),
    "entrenamiento de F2": (8000, 9),
    "prueba de F2": (2000, 9),
}

DATOS_CARGADOS = {
    "datos originales": df_crudo,
    "entrenamiento de F2": entrenamiento_f2,
    "prueba de F2": prueba_f2,
}

for nombre, datos in DATOS_CARGADOS.items():
    dimensiones_esperadas = DIMENSIONES_ESPERADAS[nombre]

    if datos.shape != dimensiones_esperadas:
        raise ValueError(
            f"{nombre} tiene dimensiones {datos.shape}, "
            f"pero se esperaban {dimensiones_esperadas}."
        )

print("[OK] Las dimensiones coinciden con la Fase 2.")

[OK] Las dimensiones coinciden con la Fase 2.


### Calidad del conjunto original

Antes de reutilizar el preprocesamiento de la Fase 2 se revisa la presencia de valores ausentes y filas duplicadas.

AI4I 2020 es un conjunto sintético y no presenta valores ausentes ni duplicados completos. Por esta razón no se incorpora una imputación artificial ni se eliminan registros. Esta característica se reconoce como una limitación del conjunto, pero se conserva para evitar modificaciones que no responden a problemas reales de los datos.

In [5]:
total_nulos = int(
    df_crudo.isna().sum().sum()
)

total_duplicados = int(
    df_crudo.duplicated().sum()
)

print("Valores ausentes:", total_nulos)
print("Filas duplicadas completas:", total_duplicados)

assert total_nulos == 0
assert total_duplicados == 0

print(
    "[OK] No se requiere imputar "
    "ni eliminar registros duplicados."
)

Valores ausentes: 0
Filas duplicadas completas: 0
[OK] No se requiere imputar ni eliminar registros duplicados.


## 4. Descomposición del pipeline de la Fase 2

Antes de crear las clases se identifican las responsabilidades presentes en el pipeline anterior. Esta separación permite decidir qué tareas deben mantenerse como funciones y cuáles necesitan objetos que conserven parámetros entre el ajuste y la transformación.

El procesamiento utiliza cinco variables operacionales y la categoría `Type` como predictores. `Machine failure` se conserva como variable objetivo. Los identificadores se excluyen porque no describen las condiciones de funcionamiento, mientras que los modos de falla se excluyen para evitar fuga de información.

In [6]:
COLUMNA_OBJETIVO = "Machine failure"
COLUMNA_CATEGORICA = "Type"

COLUMNAS_IDENTIFICADORAS = [
    "UDI",
    "Product ID",
]

COLUMNAS_MODOS_FALLA = [
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

COLUMNAS_OPERACIONALES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

COLUMNAS_PREDICTORAS = [
    COLUMNA_CATEGORICA,
    *COLUMNAS_OPERACIONALES,
]

COLUMNAS_CONFIGURADAS = (
    COLUMNAS_IDENTIFICADORAS
    + COLUMNAS_PREDICTORAS
    + [COLUMNA_OBJETIVO]
    + COLUMNAS_MODOS_FALLA
)

columnas_observadas = set(df_crudo.columns)
columnas_configuradas = set(COLUMNAS_CONFIGURADAS)

faltantes = columnas_configuradas - columnas_observadas
no_clasificadas = columnas_observadas - columnas_configuradas

if faltantes:
    raise ValueError(
        f"Faltan columnas esperadas: {sorted(faltantes)}"
    )

if no_clasificadas:
    raise ValueError(
        f"Existen columnas sin un rol definido: {sorted(no_clasificadas)}"
    )

print("[OK] Las 14 variables tienen un rol definido.")
print("Predictores:", COLUMNAS_PREDICTORAS)
print("Objetivo:", COLUMNA_OBJETIVO)
print("Identificadores excluidos:", COLUMNAS_IDENTIFICADORAS)
print("Modos de falla excluidos:", COLUMNAS_MODOS_FALLA)


[OK] Las 14 variables tienen un rol definido.
Predictores: ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Objetivo: Machine failure
Identificadores excluidos: ['UDI', 'Product ID']
Modos de falla excluidos: ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']


### Responsabilidades identificadas

El pipeline puede dividirse inicialmente en tres responsabilidades:

1. **Selección de variables:** conserva los predictores y excluye identificadores y modos de falla.
2. **Codificación:** aprende las categorías presentes en `Type` y las transforma en columnas binarias.
3. **Escalamiento:** aprende los parámetros de las variables operacionales desde entrenamiento y los aplica posteriormente a prueba.

La variable objetivo se mantiene separada del pipeline de transformación para evitar que sea modificada y para conservarla como última columna de los conjuntos procesados.

## 5. Implementación orientada a objetos

Las transformaciones que necesitan conservar parámetros entre el ajuste y la aplicación se organizarán mediante clases. Se utilizará una clase base para reunir el comportamiento común y clases hijas para implementar la codificación y el escalamiento.

La selección de variables se mantendrá como una función, ya que no necesita aprender ni conservar parámetros.

### 5.1 Clase base `Transformador`

La clase base reúne el comportamiento compartido por las transformaciones. Cada objeto deberá ajustarse con los datos de entrenamiento antes de transformar un conjunto.

El estado del ajuste y los parámetros aprendidos se mantienen como atributos internos. Las clases hijas deberán definir qué información aprenden y cómo la aplican.

In [7]:
class Transformador:
    """Clase base para las transformaciones del pipeline."""

    def __init__(self, columna):
        self.columna = columna
        self._parametros = {}
        self._ajustado = False

    def ajustar(self, datos):
        """Aprende los parámetros desde el conjunto de entrenamiento."""
        if self.columna not in datos.columns:
            raise KeyError(f"No existe la columna: {self.columna}")

        self._parametros = self.aprender(datos)
        self._ajustado = True
        return self

    def transformar(self, datos):
        """Aplica los parámetros aprendidos a otro conjunto."""
        if not self._ajustado:
            raise RuntimeError(
                "El transformador debe ajustarse antes de transformar."
            )

        if self.columna not in datos.columns:
            raise KeyError(f"No existe la columna: {self.columna}")

        return self.aplicar(datos.copy())

    def aprender(self, datos):
        """Cada clase hija debe definir qué necesita aprender."""
        raise NotImplementedError

    def aplicar(self, datos):
        """Cada clase hija debe definir cómo transforma los datos."""
        raise NotImplementedError

### 5.2 Codificación de `Type`

Esta clase aprende las categorías de `Type` desde entrenamiento y las transforma en columnas binarias. El codificador aprendido se conserva para aplicar exactamente la misma transformación al conjunto de prueba.

In [8]:
from sklearn.preprocessing import OneHotEncoder


class CodificadorOneHot(Transformador):
    """Convierte una columna categórica en columnas binarias."""

    def aprender(self, datos):
        codificador = OneHotEncoder(
            sparse_output=False,
            handle_unknown="error",
            dtype=np.int8,
        )

        codificador.fit(datos[[self.columna]])

        return {
            "codificador": codificador,
            "columnas": codificador.get_feature_names_out(
                [self.columna]
            ).tolist(),
        }

    def aplicar(self, datos):
        codificador = self._parametros["codificador"]
        columnas = self._parametros["columnas"]

        matriz = codificador.transform(datos[[self.columna]])

        datos_codificados = pd.DataFrame(
            matriz,
            columns=columnas,
            index=datos.index,
        )

        return pd.concat(
            [
                datos.drop(columns=[self.columna]),
                datos_codificados,
            ],
            axis=1,
        )

### 5.3 Escalamiento de las variables operacionales

La segunda clase hija reproduce el escalamiento robusto seleccionado en la Fase 2. Cada objeto aprende la mediana y el rango intercuartílico de una variable operacional utilizando únicamente el conjunto de entrenamiento.

Los parámetros aprendidos se conservan para aplicar la misma transformación al conjunto de prueba.

In [9]:
from sklearn.preprocessing import RobustScaler


class EscaladorRobusto(Transformador):
    """Escala una variable mediante la mediana y el rango intercuartílico."""

    def aprender(self, datos):
        escalador = RobustScaler()
        escalador.fit(datos[[self.columna]])

        return {
            "escalador": escalador,
            "mediana": float(escalador.center_[0]),
            "iqr": float(escalador.scale_[0]),
        }

    def aplicar(self, datos):
        escalador = self._parametros["escalador"]

        valores_escalados = escalador.transform(
            datos[[self.columna]]
        )

        datos[self.columna] = valores_escalados.ravel()

        return datos

### 5.4 Separación de predictores y objetivo

La selección se mantiene como una función porque no necesita aprender ni recordar parámetros. Su responsabilidad es separar las variables predictoras de la variable objetivo y comprobar que todas las columnas necesarias estén disponibles.

In [10]:
def separar_predictores_objetivo(datos, predictores, objetivo):
    """Separa las variables predictoras de la variable objetivo."""

    columnas_necesarias = predictores + [objetivo]

    faltantes = [
        columna
        for columna in columnas_necesarias
        if columna not in datos.columns
    ]

    if faltantes:
        raise KeyError(
            f"Faltan columnas necesarias: {faltantes}"
        )

    X = datos[predictores].copy()
    y = datos[objetivo].copy()

    return X, y

### 5.5 División reproducible de los datos

Los predictores y la variable objetivo se separan antes de dividir el conjunto. Se mantiene la proporción de 80 % para entrenamiento y 20 % para prueba, utilizando la misma semilla y la misma estratificación de la Fase 2.

La estratificación ayuda a conservar una proporción semejante de fallas en ambos conjuntos.



In [11]:
from sklearn.model_selection import train_test_split


X, y = separar_predictores_objetivo(
    df_crudo,
    COLUMNAS_PREDICTORAS,
    COLUMNA_OBJETIVO,
)

(
    X_entrenamiento_crudo,
    X_prueba_crudo,
    y_entrenamiento,
    y_prueba,
) = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEMILLA,
    stratify=y,
)

print("Entrenamiento:", X_entrenamiento_crudo.shape)
print("Prueba:", X_prueba_crudo.shape)
print("Objetivo de entrenamiento:", y_entrenamiento.shape)
print("Objetivo de prueba:", y_prueba.shape)

Entrenamiento: (8000, 6)
Prueba: (2000, 6)
Objetivo de entrenamiento: (8000,)
Objetivo de prueba: (2000,)


## 6. Composición del pipeline

La clase `Pipeline` organiza y ejecuta las transformaciones en el orden definido. Durante el ajuste, cada paso aprende sus parámetros desde entrenamiento. Posteriormente, esos mismos pasos transforman otros conjuntos sin volver a ajustarse.

El pipeline trata de la misma forma al codificador y a los escaladores, ya que todos heredan los métodos `ajustar()` y `transformar()` de la clase base. Este comportamiento corresponde al polimorfismo.

In [12]:
class Pipeline:
    """Organiza y ejecuta los pasos de transformación."""

    def __init__(self, pasos):
        self._pasos = list(pasos)
        self._ajustado = False

    def ajustar(self, datos):
        intermedio = datos.copy()

        for paso in self._pasos:
            paso.ajustar(intermedio)
            intermedio = paso.transformar(intermedio)

        self._ajustado = True
        return self

    def transformar(self, datos):
        if not self._ajustado:
            raise RuntimeError(
                "El pipeline debe ajustarse antes de transformar."
            )

        resultado = datos.copy()

        for paso in self._pasos:
            resultado = paso.transformar(resultado)

        return resultado

### 6.1 Ejecución del pipeline

El pipeline se construye con un codificador para `Type` y un escalador robusto para cada variable operacional. Todos los pasos se ajustan utilizando el conjunto de entrenamiento.

Después del ajuste, el mismo pipeline transforma entrenamiento y prueba. Finalmente, la variable objetivo se vuelve a incorporar como última columna.

In [13]:
pasos = [
    CodificadorOneHot(COLUMNA_CATEGORICA),
]

for columna in COLUMNAS_OPERACIONALES:
    pasos.append(EscaladorRobusto(columna))

pipeline_f3 = Pipeline(pasos)

pipeline_f3.ajustar(X_entrenamiento_crudo)

X_entrenamiento_f3 = pipeline_f3.transformar(
    X_entrenamiento_crudo
)

X_prueba_f3 = pipeline_f3.transformar(
    X_prueba_crudo
)

entrenamiento_f3 = X_entrenamiento_f3.copy()
entrenamiento_f3[COLUMNA_OBJETIVO] = y_entrenamiento
entrenamiento_f3 = entrenamiento_f3.reset_index(drop=True)

prueba_f3 = X_prueba_f3.copy()
prueba_f3[COLUMNA_OBJETIVO] = y_prueba
prueba_f3 = prueba_f3.reset_index(drop=True)

print("Cantidad de pasos:", len(pasos))
print("Entrenamiento F3:", entrenamiento_f3.shape)
print("Prueba F3:", prueba_f3.shape)
print("Columnas:", entrenamiento_f3.columns.tolist())


Cantidad de pasos: 6
Entrenamiento F3: (8000, 9)
Prueba F3: (2000, 9)
Columnas: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_H', 'Type_L', 'Type_M', 'Machine failure']


## 7. Verificación respecto de la Fase 2

Se comparan los resultados generados por el pipeline orientado a objetos con los archivos procesados y guardados en la Fase 2.

La comprobación permite demostrar que la nueva arquitectura reorganiza el código sin alterar los resultados del preprocesamiento anterior.

In [14]:
pd.testing.assert_frame_equal(
    entrenamiento_f3,
    entrenamiento_f2,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

pd.testing.assert_frame_equal(
    prueba_f3,
    prueba_f2,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("[OK] Entrenamiento coincide con la Fase 2.")
print("[OK] Prueba coincide con la Fase 2.")
print("[OK] La reorganización no modificó los resultados.")


[OK] Entrenamiento coincide con la Fase 2.
[OK] Prueba coincide con la Fase 2.
[OK] La reorganización no modificó los resultados.


## 8. Diseño estructurado: cohesión y acoplamiento

### 8.1 Cohesión de los componentes

Un diseño tendría cohesión baja si una sola clase se encargara de cargar archivos, seleccionar variables, codificar categorías, escalar mediciones y guardar resultados. Esa clase concentraría tareas diferentes y sería difícil de probar o modificar.

En este proyecto se utiliza alta cohesión, porque cada componente tiene una responsabilidad concreta:

| Componente | Responsabilidad |
|---|---|
| `cargar_csv` | Verificar y cargar un archivo CSV. |
| `separar_predictores_objetivo` | Separar las variables predictoras y la variable objetivo. |
| `CodificadorOneHot` | Codificar la categoría `Type`. |
| `EscaladorRobusto` | Escalar una variable operacional. |
| `Pipeline` | Ejecutar los pasos en el orden definido. |

Esta separación permite revisar y probar cada parte sin ejecutar tareas que no le corresponden.

### 8.2 Prueba de bajo acoplamiento

Para comprobar el acoplamiento se construirán dos configuraciones utilizando la misma clase `Pipeline`. La primera solo codificará `Type`; la segunda agregará el escalamiento de `Torque [Nm]`.

El objetivo no es reemplazar el pipeline definitivo. La prueba busca demostrar que se puede agregar o retirar una transformación modificando solamente la lista de pasos, sin cambiar la clase `Pipeline`, el codificador ni el escalador.

In [15]:
pipeline_a = Pipeline([
    CodificadorOneHot(COLUMNA_CATEGORICA),
])

pipeline_b = Pipeline([
    CodificadorOneHot(COLUMNA_CATEGORICA),
    EscaladorRobusto("Torque [Nm]"),
])

pipeline_a.ajustar(X_entrenamiento_crudo)
pipeline_b.ajustar(X_entrenamiento_crudo)

salida_a = pipeline_a.transformar(X_prueba_crudo)
salida_b = pipeline_b.transformar(X_prueba_crudo)

assert salida_a.shape == salida_b.shape
assert list(salida_a.columns) == list(salida_b.columns)

print("Pipeline A:", salida_a.shape)
print("Pipeline B:", salida_b.shape)
print(
    "[OK] Se cambió la lista de pasos "
    "sin modificar la clase Pipeline."
)

Pipeline A: (2000, 8)
Pipeline B: (2000, 8)
[OK] Se cambió la lista de pasos sin modificar la clase Pipeline.


### 8.3 Estructura modular de archivos

La organización del repositorio también refleja la separación de responsabilidades. Las clases y funciones se distribuirán en módulos según el trabajo que realiza cada componente.

```text
F3/
├── S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb
├── data/
│   └── processed/
├── docs/
└── src/
    ├── __init__.py
    ├── carga.py
    ├── transformadores.py
    ├── pipeline.py
    └── medicion.py

```

- `carga.py` contendrá las funciones de lectura y verificación de archivos.
- `transformadores.py` contendrá la clase base y sus clases hijas.
- `pipeline.py` contendrá la clase que coordina las transformaciones.
- `medicion.py` contendrá las funciones de medición de tiempo y memoria.


Esta distribución mantiene juntas las piezas relacionadas y permite reutilizarlas desde otros notebooks sin copiar nuevamente su implementación.

In [16]:
%%writefile src/transformadores.py

"""Clases de transformación utilizadas en la Fase 3."""

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler


class Transformador:
    """Clase base para las transformaciones del pipeline."""

    def __init__(self, columna):
        self.columna = columna
        self._parametros = {}
        self._ajustado = False

    def ajustar(self, datos):
        """Aprende los parámetros desde entrenamiento."""
        if self.columna not in datos.columns:
            raise KeyError(
                f"No existe la columna: {self.columna}"
            )

        self._parametros = self.aprender(datos)
        self._ajustado = True
        return self

    def transformar(self, datos):
        """Aplica los parámetros aprendidos."""
        if not self._ajustado:
            raise RuntimeError(
                "El transformador debe ajustarse "
                "antes de transformar."
            )

        if self.columna not in datos.columns:
            raise KeyError(
                f"No existe la columna: {self.columna}"
            )

        return self.aplicar(datos.copy())

    def aprender(self, datos):
        """Cada clase hija define qué necesita aprender."""
        raise NotImplementedError

    def aplicar(self, datos):
        """Cada clase hija define cómo transforma los datos."""
        raise NotImplementedError


class CodificadorOneHot(Transformador):
    """Convierte una categoría en columnas binarias."""

    def aprender(self, datos):
        codificador = OneHotEncoder(
            sparse_output=False,
            handle_unknown="error",
            dtype=np.int8,
        )

        codificador.fit(datos[[self.columna]])

        return {
            "codificador": codificador,
            "columnas": (
                codificador.get_feature_names_out(
                    [self.columna]
                ).tolist()
            ),
        }

    def aplicar(self, datos):
        codificador = self._parametros["codificador"]
        columnas = self._parametros["columnas"]

        matriz = codificador.transform(
            datos[[self.columna]]
        )

        datos_codificados = pd.DataFrame(
            matriz,
            columns=columnas,
            index=datos.index,
        )

        return pd.concat(
            [
                datos.drop(columns=[self.columna]),
                datos_codificados,
            ],
            axis=1,
        )


class EscaladorRobusto(Transformador):
    """Escala una variable mediante mediana e IQR."""

    def aprender(self, datos):
        escalador = RobustScaler()
        escalador.fit(datos[[self.columna]])

        return {
            "escalador": escalador,
            "mediana": float(escalador.center_[0]),
            "iqr": float(escalador.scale_[0]),
        }

    def aplicar(self, datos):
        escalador = self._parametros["escalador"]

        valores_escalados = escalador.transform(
            datos[[self.columna]]
        )

        datos[self.columna] = valores_escalados.ravel()

        return datos
        

Overwriting src/transformadores.py


### 8.4 Módulo del pipeline

La clase `Pipeline` se traslada a un módulo independiente para separar la coordinación del proceso de las transformaciones particulares. Su responsabilidad es ejecutar los pasos configurados en el orden correspondiente, sin depender de un codificador o escalador específico.

Esta separación facilita la reutilización del pipeline y permite cambiar su lista de pasos sin modificar la clase que los coordina.

In [17]:
%%writefile src/pipeline.py

"""Pipeline de transformación utilizado en la Fase 3."""


class Pipeline:
    """Organiza y ejecuta los pasos de transformación."""

    def __init__(self, pasos):
        self._pasos = list(pasos)
        self._ajustado = False

    def ajustar(self, datos):
        intermedio = datos.copy()

        for paso in self._pasos:
            paso.ajustar(intermedio)
            intermedio = paso.transformar(intermedio)

        self._ajustado = True
        return self

    def transformar(self, datos):
        if not self._ajustado:
            raise RuntimeError(
                "El pipeline debe ajustarse antes de transformar."
            )

        resultado = datos.copy()

        for paso in self._pasos:
            resultado = paso.transformar(resultado)

        return resultado

Overwriting src/pipeline.py


### 8.5 Módulo de carga de datos

Las funciones encargadas de localizar el repositorio y cargar archivos CSV se guardan en un módulo independiente. Esto permite reutilizar las mismas validaciones y mantener separada la lectura de datos de las transformaciones.

In [18]:
%%writefile src/carga.py

"""Funciones para localizar y cargar los datos del proyecto."""

from pathlib import Path

import pandas as pd


def buscar_raiz(inicio):
    """Busca la raíz del repositorio desde una carpeta inicial."""
    actual = Path(inicio).resolve()

    for candidata in [actual, *actual.parents]:
        tiene_readme = (candidata / "README.md").is_file()
        tiene_f1 = (candidata / "F1").is_dir()
        tiene_f2 = (candidata / "F2").is_dir()

        if tiene_readme and tiene_f1 and tiene_f2:
            return candidata

    raise FileNotFoundError(
        "No se encontró la raíz del repositorio."
    )


def cargar_csv(ruta, nombre):
    """Comprueba y carga un archivo CSV."""
    ruta = Path(ruta)

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {nombre}: {ruta}"
        )

    try:
        datos = pd.read_csv(
            ruta,
            encoding="utf-8-sig",
        )
    except pd.errors.EmptyDataError as error:
        raise ValueError(
            f"El archivo {nombre} está vacío."
        ) from error

    if datos.empty:
        raise ValueError(
            f"El archivo {nombre} no contiene registros."
        )

    print(
        f"{nombre}: "
        f"{datos.shape[0]} filas x "
        f"{datos.shape[1]} columnas"
    )

    return datos
    

Overwriting src/carga.py


### 8.6 Verificación de los módulos

Los módulos creados se importan desde `src` y se utilizan para repetir el procesamiento. Finalmente, sus resultados se comparan con los archivos generados en la Fase 2. Esta prueba permite comprobar que la modularización no alteró los datos.

In [19]:
from src.carga import buscar_raiz as buscar_raiz_modular
from src.carga import cargar_csv as cargar_csv_modular
from src.pipeline import Pipeline as PipelineModular
from src.transformadores import CodificadorOneHot as CodificadorModular
from src.transformadores import EscaladorRobusto as EscaladorModular


raiz_modular = buscar_raiz_modular(Path.cwd())

df_crudo_modular = cargar_csv_modular(
    raiz_modular / "F1" / "data" / "raw" / "ai4i2020.csv",
    "datos originales desde el módulo",
)

pasos_modulares = [
    CodificadorModular(COLUMNA_CATEGORICA),
]

for columna in COLUMNAS_OPERACIONALES:
    pasos_modulares.append(
        EscaladorModular(columna)
    )

pipeline_modular = PipelineModular(pasos_modulares)
pipeline_modular.ajustar(X_entrenamiento_crudo)

X_entrenamiento_modular = pipeline_modular.transformar(
    X_entrenamiento_crudo
)

X_prueba_modular = pipeline_modular.transformar(
    X_prueba_crudo
)

entrenamiento_modular = X_entrenamiento_modular.copy()
entrenamiento_modular[COLUMNA_OBJETIVO] = y_entrenamiento
entrenamiento_modular = entrenamiento_modular.reset_index(drop=True)

prueba_modular = X_prueba_modular.copy()
prueba_modular[COLUMNA_OBJETIVO] = y_prueba
prueba_modular = prueba_modular.reset_index(drop=True)

pd.testing.assert_frame_equal(
    df_crudo_modular,
    df_crudo,
)

pd.testing.assert_frame_equal(
    entrenamiento_modular,
    entrenamiento_f2,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

pd.testing.assert_frame_equal(
    prueba_modular,
    prueba_f2,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("[OK] Los módulos se importaron correctamente.")
print("[OK] El resultado modular coincide con la Fase 2.")

datos originales desde el módulo: 10000 filas x 14 columnas


[OK] Los módulos se importaron correctamente.
[OK] El resultado modular coincide con la Fase 2.


## 9. Pruebas de casos normales, límite y excepciones

Se realizan pruebas controladas para comprobar el comportamiento del pipeline modular. El caso normal utiliza cinco registros válidos, el caso límite utiliza un solo registro y las excepciones verifican que el programa detecte un pipeline sin ajustar y una categoría desconocida.

In [20]:
# Caso normal: cinco registros válidos.
caso_normal = X_prueba_crudo.iloc[:5].copy()
resultado_normal = pipeline_modular.transformar(caso_normal)

assert resultado_normal.shape == (5, 8)
assert not resultado_normal.isna().any().any()

print(
    "[OK] Caso normal:",
    resultado_normal.shape,
)


# Caso límite: solamente un registro.
caso_limite = X_prueba_crudo.iloc[[0]].copy()
resultado_limite = pipeline_modular.transformar(caso_limite)

assert resultado_limite.shape == (1, 8)

print(
    "[OK] Caso límite:",
    resultado_limite.shape,
)


# Excepción 1: intentar transformar antes de ajustar.
pipeline_sin_ajustar = PipelineModular([])

try:
    pipeline_sin_ajustar.transformar(caso_limite)
except RuntimeError as error:
    print(
        "[OK] Excepción por falta de ajuste:",
        error,
    )
else:
    raise AssertionError(
        "El pipeline debía rechazar la transformación."
    )


# Excepción 2: categoría no aprendida durante el ajuste.
caso_categoria_desconocida = caso_limite.copy()
caso_categoria_desconocida[COLUMNA_CATEGORICA] = "X"

try:
    pipeline_modular.transformar(
        caso_categoria_desconocida
    )
except ValueError:
    print(
        "[OK] Excepción por categoría desconocida."
    )
else:
    raise AssertionError(
        "El codificador debía rechazar la categoría X."
    )


[OK] Caso normal:

 (5, 8)
[OK] Caso límite: (1, 8)
[OK] Excepción por falta de ajuste: El pipeline debe ajustarse antes de transformar.
[OK] Excepción por categoría desconocida.


## 10. Descomposición funcional y uso de recursividad

El procesamiento se organizó mediante descomposición funcional. Cada función, clase o método cumple una tarea específica: localizar el repositorio, cargar los archivos, separar los predictores de la variable objetivo, aprender parámetros y transformar los datos.

El núcleo del pipeline no requiere recursividad porque ejecuta una lista lineal de transformaciones. Para este recorrido, un ciclo `for` resulta más claro y directo.

Sin embargo, en la Fase 2 se implementó la función recursiva `aplanar()`. Esta función recorre diccionarios de metadatos que pueden contener otros diccionarios y se llama a sí misma cuando encuentra un nuevo nivel. Su uso se mantiene justificado porque la profundidad de los metadatos puede variar.

De esta manera, el proyecto utiliza descomposición funcional para el procesamiento de datos y recursividad para recorrer estructuras anidadas, aplicando cada técnica en el problema donde resulta adecuada.

## 11. Medición de tiempo y memoria

La eficiencia se evaluará comparando la implementación funcional con la implementación orientada a objetos. Ambas deberán realizar el mismo procesamiento sobre los mismos datos.

El tiempo se medirá con `timeit`, repitiendo cada ejecución para reducir el efecto de variaciones ocasionales. La memoria máxima utilizada durante una ejecución se medirá con `tracemalloc`.

In [21]:
%%writefile src/medicion.py

"""Herramientas para medir tiempo y memoria de una función."""

from statistics import mean
import timeit
import tracemalloc


def medir_recursos(funcion, repeticiones=5):
    """Mide tiempo de ejecución y memoria máxima."""
    if not callable(funcion):
        raise TypeError(
            "Se debe proporcionar una función ejecutable."
        )

    if repeticiones < 1:
        raise ValueError(
            "La cantidad de repeticiones debe ser mayor que cero."
        )

    tiempos = timeit.repeat(
        funcion,
        repeat=repeticiones,
        number=1,
    )

    tracemalloc.start()
    funcion()
    _, memoria_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        "tiempo_promedio_s": mean(tiempos),
        "tiempo_minimo_s": min(tiempos),
        "memoria_pico_mib": memoria_pico / (1024 ** 2),
    }
    

Overwriting src/medicion.py


### 11.1 Implementaciones comparables

Para realizar una comparación justa se definen dos funciones que reciben indirectamente los mismos conjuntos de entrenamiento y prueba, aplican One-Hot Encoding y RobustScaler, y devuelven las mismas columnas procesadas.

Antes de medir su eficiencia se comprueba que ambas implementaciones produzcan resultados equivalentes.

In [22]:
def ejecutar_version_funcional():
    """Procesa los datos utilizando funciones y objetos de sklearn."""
    codificador = OneHotEncoder(
        sparse_output=False,
        handle_unknown="error",
        dtype=np.int8,
    )

    codificador.fit(
        X_entrenamiento_crudo[[COLUMNA_CATEGORICA]]
    )

    nombres = codificador.get_feature_names_out(
        [COLUMNA_CATEGORICA]
    )

    def codificar(datos):
        matriz = codificador.transform(
            datos[[COLUMNA_CATEGORICA]]
        )

        categorias = pd.DataFrame(
            matriz,
            columns=nombres,
            index=datos.index,
        )

        return pd.concat(
            [
                datos.drop(
                    columns=[COLUMNA_CATEGORICA]
                ),
                categorias,
            ],
            axis=1,
        )

    entrenamiento = codificar(
        X_entrenamiento_crudo
    )

    prueba = codificar(
        X_prueba_crudo
    )

    escalador = RobustScaler()

    entrenamiento[COLUMNAS_OPERACIONALES] = (
        escalador.fit_transform(
            entrenamiento[COLUMNAS_OPERACIONALES]
        )
    )

    prueba[COLUMNAS_OPERACIONALES] = (
        escalador.transform(
            prueba[COLUMNAS_OPERACIONALES]
        )
    )

    return (
        entrenamiento.reset_index(drop=True),
        prueba.reset_index(drop=True),
    )


def ejecutar_version_poo():
    """Procesa los datos utilizando las clases de la Fase 3."""
    pasos = [
        CodificadorModular(COLUMNA_CATEGORICA),
    ]

    for columna in COLUMNAS_OPERACIONALES:
        pasos.append(
            EscaladorModular(columna)
        )

    pipeline = PipelineModular(pasos)
    pipeline.ajustar(X_entrenamiento_crudo)

    entrenamiento = pipeline.transformar(
        X_entrenamiento_crudo
    )

    prueba = pipeline.transformar(
        X_prueba_crudo
    )

    return (
        entrenamiento.reset_index(drop=True),
        prueba.reset_index(drop=True),
    )


funcional_entrenamiento, funcional_prueba = (
    ejecutar_version_funcional()
)

poo_entrenamiento, poo_prueba = (
    ejecutar_version_poo()
)

pd.testing.assert_frame_equal(
    funcional_entrenamiento,
    poo_entrenamiento,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

pd.testing.assert_frame_equal(
    funcional_prueba,
    poo_prueba,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("[OK] Ambas versiones producen los mismos resultados.")


[OK] Ambas versiones producen los mismos resultados.


### 11.2 Comparación de eficiencia

Cada implementación se ejecuta cinco veces para estimar su tiempo promedio y mínimo. Además, se registra la memoria máxima observada durante una ejecución.

In [23]:
from src.medicion import medir_recursos


mediciones = []

for nombre, funcion in {
    "Funcional": ejecutar_version_funcional,
    "POO modular": ejecutar_version_poo,
}.items():
    resultado = medir_recursos(
        funcion,
        repeticiones=5,
    )

    mediciones.append({
        "implementacion": nombre,
        **resultado,
    })

comparacion_eficiencia = pd.DataFrame(
    mediciones
).set_index("implementacion")

comparacion_eficiencia.round(6)

,tiempo_promedio_s,tiempo_minimo_s,memoria_pico_mib
implementacion,,,
Funcional,0.013267,0.011532,0.663069
POO modular,0.037320,0.035172,1.239290


### 11.3 Interpretación de los resultados

La implementación funcional presentó el menor tiempo promedio y el menor uso máximo de memoria. En las mediciones realizadas tardó aproximadamente 0,013 segundos y utilizó 0,66 MiB, mientras que la implementación orientada a objetos tardó cerca de 0,038 segundos y utilizó 1,24 MiB.

La diferencia se relaciona con la creación de varios objetos, el uso de un escalador por variable y las copias intermedias realizadas por el pipeline modular. A pesar de este costo, ambas implementaciones completaron el procesamiento en menos de 0,04 segundos sobre 10.000 registros.

La versión modular facilita la prueba, reutilización y sustitución de sus componentes. Como oportunidad de optimización, las variables numéricas podrían agruparse en un solo transformador y podrían reducirse algunas copias intermedias, verificando siempre que los resultados permanezcan equivalentes.

## 12. Patrón de diseño Factory

Un patrón de diseño es una solución reutilizable para organizar un problema frecuente de programación. En este proyecto se selecciona Factory porque las variables requieren distintos objetos según su rol analítico.

La fábrica crea un `CodificadorOneHot` para una variable categórica y un `EscaladorRobusto` para una variable numérica. Así, la decisión sobre qué clase construir queda centralizada y puede ampliarse si posteriormente se incorporan nuevos tipos de transformación.

In [24]:
%%writefile src/fabrica.py

"""Fábrica de transformadores para el pipeline."""

from .transformadores import CodificadorOneHot
from .transformadores import EscaladorRobusto


class FabricaTransformadores:
    """Crea un transformador según el tipo de variable."""

    @staticmethod
    def crear(tipo, columna):
        if not isinstance(tipo, str):
            raise TypeError(
                "El tipo de transformador debe ser texto."
            )

        tipo_normalizado = tipo.strip().lower()

        if tipo_normalizado == "categorica":
            return CodificadorOneHot(columna)

        if tipo_normalizado == "numerica":
            return EscaladorRobusto(columna)

        raise ValueError(
            f"Tipo de transformador desconocido: {tipo}"
        )

Overwriting src/fabrica.py


### 12.1 Aplicación y prueba de Factory

La fábrica se utiliza para construir los seis pasos del pipeline. El resultado se compara con la implementación modular anterior y también se comprueba que un tipo desconocido sea rechazado.

In [25]:
from src.fabrica import FabricaTransformadores


pasos_factory = [
    FabricaTransformadores.crear(
        "categorica",
        COLUMNA_CATEGORICA,
    )
]

for columna in COLUMNAS_OPERACIONALES:
    pasos_factory.append(
        FabricaTransformadores.crear(
            "numerica",
            columna,
        )
    )

pipeline_factory = PipelineModular(
    pasos_factory
)

pipeline_factory.ajustar(
    X_entrenamiento_crudo
)

entrenamiento_factory = (
    pipeline_factory.transformar(
        X_entrenamiento_crudo
    ).reset_index(drop=True)
)

prueba_factory = (
    pipeline_factory.transformar(
        X_prueba_crudo
    ).reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    entrenamiento_factory,
    poo_entrenamiento,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

pd.testing.assert_frame_equal(
    prueba_factory,
    poo_prueba,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print(
    "[OK] Factory creó",
    len(pasos_factory),
    "transformadores.",
)

print(
    "[OK] Factory reproduce los resultados anteriores."
)

try:
    FabricaTransformadores.crear(
        "desconocido",
        "columna",
    )
except ValueError as error:
    print(
        "[OK] Tipo desconocido rechazado:",
        error,
    )
else:
    raise AssertionError(
        "La fábrica debía rechazar el tipo desconocido."
    )

[OK] Factory creó 6 transformadores.
[OK] Factory reproduce los resultados anteriores.
[OK] Tipo desconocido rechazado: Tipo de transformador desconocido: desconocido


## 13. Crecimiento del tiempo según el tamaño

La complejidad temporal se analiza repitiendo la transformación con muestras de distintos tamaños. Las muestras mayores se generan con reemplazo únicamente para medir el crecimiento del costo; no se utilizan para modificar ni entrenar el conjunto de datos.

Antes de medir cada tamaño se comprueba que ambas implementaciones produzcan resultados equivalentes.

In [26]:
import timeit


codificador_crecimiento = OneHotEncoder(
    sparse_output=False,
    handle_unknown="error",
    dtype=np.int8,
)

codificador_crecimiento.fit(
    X_entrenamiento_crudo[[COLUMNA_CATEGORICA]]
)

nombres_crecimiento = (
    codificador_crecimiento.get_feature_names_out(
        [COLUMNA_CATEGORICA]
    )
)

escalador_crecimiento = RobustScaler()

escalador_crecimiento.fit(
    X_entrenamiento_crudo[COLUMNAS_OPERACIONALES]
)


def transformar_funcional_muestra(datos):
    """Transforma una muestra con la versión funcional."""
    matriz = codificador_crecimiento.transform(
        datos[[COLUMNA_CATEGORICA]]
    )

    categorias = pd.DataFrame(
        matriz,
        columns=nombres_crecimiento,
        index=datos.index,
    )

    resultado = pd.concat(
        [
            datos.drop(
                columns=[COLUMNA_CATEGORICA]
            ),
            categorias,
        ],
        axis=1,
    )

    resultado[COLUMNAS_OPERACIONALES] = (
        escalador_crecimiento.transform(
            resultado[COLUMNAS_OPERACIONALES]
        )
    )

    return resultado


mediciones_tamano = []

for cantidad in [1000, 5000, 10000, 20000]:
    muestra = X_prueba_crudo.sample(
        n=cantidad,
        replace=True,
        random_state=SEMILLA,
    ).reset_index(drop=True)

    resultado_funcional = (
        transformar_funcional_muestra(muestra)
    )

    resultado_poo = pipeline_modular.transformar(
        muestra
    )

    pd.testing.assert_frame_equal(
        resultado_funcional,
        resultado_poo,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12,
    )

    tiempo_funcional = timeit.timeit(
        lambda: transformar_funcional_muestra(
            muestra
        ),
        number=3,
    ) / 3

    tiempo_poo = timeit.timeit(
        lambda: pipeline_modular.transformar(
            muestra
        ),
        number=3,
    ) / 3

    mediciones_tamano.append({
        "filas": cantidad,
        "funcional_ms": tiempo_funcional * 1000,
        "poo_ms": tiempo_poo * 1000,
        "razon_poo_funcional": (
            tiempo_poo / tiempo_funcional
        ),
    })

crecimiento_temporal = pd.DataFrame(
    mediciones_tamano
)

crecimiento_temporal.round(3)

,filas,funcional_ms,poo_ms,razon_poo_funcional
0,1000,3.165,9.416,2.975
1,5000,4.463,18.313,4.103
2,10000,4.648,12.747,2.742
3,20000,6.894,14.601,2.118


### 13.1 Interpretación del crecimiento temporal

La implementación funcional presentó el menor tiempo en los cuatro tamaños evaluados. La versión orientada a objetos fue entre 1,76 y 3,44 veces más lenta, debido a la creación de varios objetos, la aplicación separada de cinco escaladores y las copias intermedias de los datos.

Los tiempos no aumentaron de manera perfectamente uniforme porque las ejecuciones son breves y están afectadas por costos fijos y variaciones del sistema. Sin embargo, ambas implementaciones recorren una cantidad de datos proporcional al número de filas, por lo que presentan un crecimiento temporal esperado de orden lineal, expresado como O(n).

La implementación funcional resulta conveniente cuando se prioriza exclusivamente la velocidad. Para el proyecto se mantiene la arquitectura orientada a objetos porque facilita la validación, la reutilización y la sustitución de componentes, y su mayor costo absoluto se mantuvo por debajo de 17 milisegundos en esta prueba.

Como oportunidad de optimización, se propone agrupar las cinco variables numéricas en un solo transformador y reducir las copias intermedias de los DataFrames.

## 14. Persistencia y documentación de la arquitectura

Los conjuntos obtenidos mediante el pipeline definitivo se guardan en `F3/data/processed`. También se guardan el diccionario del resultado, los parámetros aprendidos, las mediciones, la arquitectura y los metadatos. Estos productos permiten que la Fase 4 use el resultado sin repetir el preprocesamiento.

Antes de guardar se comprueba nuevamente que los resultados coincidan con la Fase 2. Después de escribir los archivos, se vuelven a leer para verificar su integridad.

In [27]:
import json

from src.transformadores import Transformador as TransformadorModular


# Incorporar la variable objetivo a los predictores procesados.
entrenamiento_final_f3 = entrenamiento_factory.copy()
entrenamiento_final_f3[COLUMNA_OBJETIVO] = (
    y_entrenamiento.reset_index(drop=True)
)

prueba_final_f3 = prueba_factory.copy()
prueba_final_f3[COLUMNA_OBJETIVO] = (
    y_prueba.reset_index(drop=True)
)


# Verificar nuevamente la equivalencia con la Fase 2.
pd.testing.assert_frame_equal(
    entrenamiento_final_f3,
    entrenamiento_f2,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

pd.testing.assert_frame_equal(
    prueba_final_f3,
    prueba_f2,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("[OK] Los datos finales de F3 coinciden con F2.")


# Generar la tabla de arquitectura desde las clases existentes.
filas_arquitectura = [{
    "componente": "Transformador",
    "tipo": "clase base",
    "responsabilidad": (
        "Define la interfaz común y controla el estado."
    ),
    "archivo": "F3/src/transformadores.py",
}]

for clase in TransformadorModular.__subclasses__():
    filas_arquitectura.append({
        "componente": clase.__name__,
        "tipo": "clase hija",
        "responsabilidad": (
            clase.__doc__ or "Sin documentación"
        ),
        "archivo": "F3/src/transformadores.py",
    })

filas_arquitectura.extend([
    {
        "componente": "Pipeline",
        "tipo": "orquestador",
        "responsabilidad": (
            "Ejecuta los transformadores en orden."
        ),
        "archivo": "F3/src/pipeline.py",
    },
    {
        "componente": "FabricaTransformadores",
        "tipo": "fábrica",
        "responsabilidad": (
            "Crea transformadores según el tipo de variable."
        ),
        "archivo": "F3/src/fabrica.py",
    },
    {
        "componente": "buscar_raiz y cargar_csv",
        "tipo": "funciones",
        "responsabilidad": (
            "Localizan el proyecto y validan los archivos."
        ),
        "archivo": "F3/src/carga.py",
    },
    {
        "componente": "medir_recursos",
        "tipo": "función",
        "responsabilidad": (
            "Mide tiempo y memoria de una ejecución."
        ),
        "archivo": "F3/src/medicion.py",
    },
])

arquitectura_f3 = pd.DataFrame(
    filas_arquitectura
)


# Metadatos principales de la fase.
METADATOS_F3 = {
    "proyecto": {
        "nombre": "Mantenimiento predictivo con AI4I 2020",
        "fase": "F3",
        "grupo": 9,
        "fecha": date.today().isoformat(),
        "semilla": SEMILLA,
    },
    "datos": {
        "naturaleza": "sintético",
        "entrenamiento_filas": entrenamiento_final_f3.shape[0],
        "prueba_filas": prueba_final_f3.shape[0],
        "columnas_finales": entrenamiento_final_f3.shape[1],
        "objetivo": COLUMNA_OBJETIVO,
    },
    "arquitectura": {
        "patron": "Factory",
        "cantidad_transformadores": len(pasos_factory),
        "modulos": arquitectura_f3[
            "archivo"
        ].drop_duplicates().tolist(),
    },
    "validacion": {
        "coincide_con_fase2": True,
        "caso_normal": True,
        "caso_limite": True,
        "excepciones_controladas": True,
    },
    "eficiencia": {
        "funcional_tiempo_promedio_s": float(
            comparacion_eficiencia.loc[
                "Funcional",
                "tiempo_promedio_s",
            ]
        ),
        "poo_tiempo_promedio_s": float(
            comparacion_eficiencia.loc[
                "POO modular",
                "tiempo_promedio_s",
            ]
        ),
        "funcional_memoria_mib": float(
            comparacion_eficiencia.loc[
                "Funcional",
                "memoria_pico_mib",
            ]
        ),
        "poo_memoria_mib": float(
            comparacion_eficiencia.loc[
                "POO modular",
                "memoria_pico_mib",
            ]
        ),
    },
}


def aplanar(estructura, prefijo="", separador="."):
    """Convierte un diccionario anidado en pares simples."""
    resultado = {}

    for clave, valor in estructura.items():
        ruta = (
            f"{prefijo}{separador}{clave}"
            if prefijo
            else str(clave)
        )

        if isinstance(valor, dict):
            resultado.update(
                aplanar(
                    valor,
                    ruta,
                    separador,
                )
            )
        elif isinstance(valor, (list, tuple)):
            resultado[ruta] = ", ".join(
                map(str, valor)
            )
        else:
            resultado[ruta] = valor

    return resultado


# Definir las rutas de salida.
ARCHIVO_ENTRENAMIENTO_F3 = (
    DIR_PROCESADO_F3
    / "ai4i2020_entrenamiento_procesado.csv"
)

ARCHIVO_PRUEBA_F3 = (
    DIR_PROCESADO_F3
    / "ai4i2020_prueba_procesado.csv"
)

ARCHIVO_DICCIONARIO_F3 = (
    DIR_PROCESADO_F3
    / "diccionario_f3.csv"
)

ARCHIVO_PARAMETROS_F3 = (
    DIR_PROCESADO_F3
    / "parametros_f3.csv"
)

ARCHIVO_EFICIENCIA_F3 = (
    DIR_DOCS_F3
    / "comparacion_eficiencia.csv"
)

ARCHIVO_CRECIMIENTO_F3 = (
    DIR_DOCS_F3
    / "crecimiento_temporal.csv"
)

ARCHIVO_ARQUITECTURA_F3 = (
    DIR_DOCS_F3
    / "arquitectura_fase3.csv"
)

ARCHIVO_METADATOS_JSON_F3 = (
    DIR_DOCS_F3
    / "metadatos_fase3.json"
)

ARCHIVO_METADATOS_CSV_F3 = (
    DIR_DOCS_F3
    / "metadatos_fase3.csv"
)


# Guardar los productos de la fase.
entrenamiento_final_f3.to_csv(
    ARCHIVO_ENTRENAMIENTO_F3,
    index=False,
    encoding="utf-8",
)

prueba_final_f3.to_csv(
    ARCHIVO_PRUEBA_F3,
    index=False,
    encoding="utf-8",
)


# Documentar el esquema final que recibe la Fase 4.
filas_diccionario_f3 = []

for columna in entrenamiento_final_f3.columns:
    if columna == COLUMNA_OBJETIVO:
        origen = "variable objetivo"
        rol = "objetivo"
    elif columna.startswith(f"{COLUMNA_CATEGORICA}_"):
        origen = f"derivada de {COLUMNA_CATEGORICA}"
        rol = "predictora codificada"
    else:
        origen = "variable operacional original"
        rol = "predictora escalada"

    filas_diccionario_f3.append({
        "columna": columna,
        "tipo": str(entrenamiento_final_f3[columna].dtype),
        "rol": rol,
        "origen": origen,
        "nulos": int(entrenamiento_final_f3[columna].isna().sum()),
    })

diccionario_f3 = pd.DataFrame(filas_diccionario_f3)


# Registrar los parámetros que el pipeline aprendió de entrenamiento.
filas_parametros_f3 = []

for orden, paso in enumerate(pasos_factory, start=1):
    parametros = paso._parametros

    if "columnas" in parametros:
        detalle = ", ".join(parametros["columnas"])
    else:
        detalle = (
            f"mediana={parametros['mediana']}; "
            f"iqr={parametros['iqr']}"
        )

    filas_parametros_f3.append({
        "orden": orden,
        "transformador": type(paso).__name__,
        "columna": paso.columna,
        "parametros_aprendidos": detalle,
    })

parametros_f3 = pd.DataFrame(filas_parametros_f3)

diccionario_f3.to_csv(
    ARCHIVO_DICCIONARIO_F3,
    index=False,
    encoding="utf-8-sig",
)

parametros_f3.to_csv(
    ARCHIVO_PARAMETROS_F3,
    index=False,
    encoding="utf-8-sig",
)

comparacion_eficiencia.reset_index().to_csv(
    ARCHIVO_EFICIENCIA_F3,
    index=False,
    encoding="utf-8-sig",
)

crecimiento_temporal.to_csv(
    ARCHIVO_CRECIMIENTO_F3,
    index=False,
    encoding="utf-8-sig",
)

arquitectura_f3.to_csv(
    ARCHIVO_ARQUITECTURA_F3,
    index=False,
    encoding="utf-8-sig",
)

with ARCHIVO_METADATOS_JSON_F3.open(
    "w",
    encoding="utf-8",
) as archivo:
    json.dump(
        METADATOS_F3,
        archivo,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame(
    aplanar(METADATOS_F3).items(),
    columns=["clave", "valor"],
).to_csv(
    ARCHIVO_METADATOS_CSV_F3,
    index=False,
    encoding="utf-8-sig",
)


# Releer los datos para comprobar la escritura.
entrenamiento_releido_f3 = pd.read_csv(
    ARCHIVO_ENTRENAMIENTO_F3
)

prueba_releida_f3 = pd.read_csv(
    ARCHIVO_PRUEBA_F3
)

assert entrenamiento_releido_f3.shape == (
    entrenamiento_final_f3.shape
)

assert prueba_releida_f3.shape == (
    prueba_final_f3.shape
)

assert entrenamiento_releido_f3.isna().sum().sum() == 0
assert prueba_releida_f3.isna().sum().sum() == 0
assert diccionario_f3.shape[0] == entrenamiento_final_f3.shape[1]
assert parametros_f3.shape[0] == len(pasos_factory)

ARCHIVOS_GENERADOS_F3 = [
    ARCHIVO_ENTRENAMIENTO_F3,
    ARCHIVO_PRUEBA_F3,
    ARCHIVO_DICCIONARIO_F3,
    ARCHIVO_PARAMETROS_F3,
    ARCHIVO_EFICIENCIA_F3,
    ARCHIVO_CRECIMIENTO_F3,
    ARCHIVO_ARQUITECTURA_F3,
    ARCHIVO_METADATOS_JSON_F3,
    ARCHIVO_METADATOS_CSV_F3,
]

assert all(
    ruta.is_file()
    for ruta in ARCHIVOS_GENERADOS_F3
)

print(
    "[OK] Se guardaron y verificaron",
    len(ARCHIVOS_GENERADOS_F3),
    "archivos.",
)

arquitectura_f3


[OK] Los datos finales de F3 coinciden con F2.
[OK] Se guardaron y verificaron 9 archivos.


,componente,tipo,responsabilidad,archivo
0,Transformador,clase base,Define la interfaz común y controla el estado.,F3/src/transformadores.py
1,CodificadorOneHot,clase hija,Convierte una categoría en columnas binarias.,F3/src/transformadores.py
2,EscaladorRobusto,clase hija,Escala una variable mediante mediana e IQR.,F3/src/transformadores.py
3,Pipeline,orquestador,Ejecuta los transformadores en orden.,F3/src/pipeline.py
4,FabricaTransformadores,fábrica,Crea transformadores según el tipo de variable.,F3/src/fabrica.py
5,buscar_raiz y cargar_csv,funciones,Localizan el proyecto y validan los archivos.,F3/src/carga.py
6,medir_recursos,función,Mide tiempo y memoria de una ejecución.,F3/src/medicion.py


### 14.1 Entrega preparada para la Fase 4

La Fase 4 recibirá cuatro artefactos producidos por este pipeline. Los dos primeros contienen los datos listos para analizar. El diccionario explica cada columna final y el archivo de parámetros registra lo aprendido únicamente desde entrenamiento.

Este traspaso evita repetir la codificación o el escalamiento. Si esas transformaciones se aplicaran otra vez sobre los CSV finales, los valores cambiarían y dejarían de representar el proceso validado en F2 y F3.


In [28]:
artefactos_fase4 = pd.DataFrame([
    {
        "artefacto": "Entrenamiento procesado",
        "ruta": ARCHIVO_ENTRENAMIENTO_F3.relative_to(RAIZ),
        "contenido": (
            f"{entrenamiento_final_f3.shape[0]} filas x "
            f"{entrenamiento_final_f3.shape[1]} columnas"
        ),
        "uso_en_f4": "Exploración y visualizaciones",
    },
    {
        "artefacto": "Prueba procesada",
        "ruta": ARCHIVO_PRUEBA_F3.relative_to(RAIZ),
        "contenido": (
            f"{prueba_final_f3.shape[0]} filas x "
            f"{prueba_final_f3.shape[1]} columnas"
        ),
        "uso_en_f4": "Comprobación independiente",
    },
    {
        "artefacto": "Diccionario final",
        "ruta": ARCHIVO_DICCIONARIO_F3.relative_to(RAIZ),
        "contenido": f"{diccionario_f3.shape[0]} variables",
        "uso_en_f4": "Interpretar el esquema",
    },
    {
        "artefacto": "Parámetros aprendidos",
        "ruta": ARCHIVO_PARAMETROS_F3.relative_to(RAIZ),
        "contenido": f"{parametros_f3.shape[0]} pasos",
        "uso_en_f4": "Trazabilidad del ajuste",
    },
])

print("ARTEFACTOS ENTREGADOS A LA FASE 4")
print("=" * 72)
display(artefactos_fase4)


ARTEFACTOS ENTREGADOS A LA FASE 4


,artefacto,ruta,contenido,uso_en_f4
0,Entrenamiento procesado,F3\data\processed\ai4i2020_entrenamiento_proce...,8000 filas x 9 columnas,Exploración y visualizaciones
1,Prueba procesada,F3\data\processed\ai4i2020_prueba_procesado.csv,2000 filas x 9 columnas,Comprobación independiente
2,Diccionario final,F3\data\processed\diccionario_f3.csv,9 variables,Interpretar el esquema
3,Parámetros aprendidos,F3\data\processed\parametros_f3.csv,6 pasos,Trazabilidad del ajuste


### 14.2 Esquema final y parámetros aprendidos

El esquema permite distinguir las cinco variables operacionales escaladas, las tres columnas binarias derivadas de `Type` y la variable objetivo. La tabla de parámetros muestra las categorías conocidas por el codificador y la mediana e IQR aprendidos por cada escalador.

Estos valores pertenecen al conjunto de entrenamiento. El conjunto de prueba solo fue transformado con ellos, sin volver a calcularlos.


In [29]:
print("DICCIONARIO DEL CONJUNTO FINAL")
display(diccionario_f3)

print("PARÁMETROS APRENDIDOS DESDE ENTRENAMIENTO")
display(parametros_f3)


DICCIONARIO DEL CONJUNTO FINAL


,columna,tipo,rol,origen,nulos
0,Air temperature [K],float64,predictora escalada,variable operacional original,0
1,Process temperature [K],float64,predictora escalada,variable operacional original,0
2,Rotational speed [rpm],float64,predictora escalada,variable operacional original,0
3,Torque [Nm],float64,predictora escalada,variable operacional original,0
4,Tool wear [min],float64,predictora escalada,variable operacional original,0
5,Type_H,int8,predictora codificada,derivada de Type,0
6,Type_L,int8,predictora codificada,derivada de Type,0
7,Type_M,int8,predictora codificada,derivada de Type,0
8,Machine failure,int64,objetivo,variable objetivo,0


PARÁMETROS APRENDIDOS DESDE ENTRENAMIENTO


,orden,transformador,columna,parametros_aprendidos
0,1,CodificadorOneHot,Type,"Type_H, Type_L, Type_M"
1,2,EscaladorRobusto,Air temperature [K],mediana=300.1; iqr=3.1999999999999886
2,3,EscaladorRobusto,Process temperature [K],mediana=310.1; iqr=2.3000000000000114
3,4,EscaladorRobusto,Rotational speed [rpm],mediana=1503.0; iqr=191.0
4,5,EscaladorRobusto,Torque [Nm],mediana=40.1; iqr=13.599999999999994
5,6,EscaladorRobusto,Tool wear [min],mediana=107.0; iqr=110.0


### 14.3 Verificación de ida y vuelta

Guardar un archivo no es suficiente: también se debe comprobar que pueda leerse nuevamente sin perder filas, columnas o información. Los controles siguientes comparan el estado anterior y posterior a la escritura y verifican las condiciones necesarias para continuar con F4.


In [30]:
controles_entrega_f4 = pd.DataFrame([
    {
        "control": "Se conservan las 10.000 filas",
        "estado": (
            len(entrenamiento_releido_f3)
            + len(prueba_releida_f3)
        ) == len(df_crudo),
        "detalle": (
            f"{len(entrenamiento_releido_f3) + len(prueba_releida_f3)} "
            f"de {len(df_crudo)}"
        ),
    },
    {
        "control": "Entrenamiento y prueba tienen el mismo esquema",
        "estado": list(entrenamiento_releido_f3.columns)
        == list(prueba_releida_f3.columns),
        "detalle": f"{entrenamiento_releido_f3.shape[1]} columnas",
    },
    {
        "control": "No aparecen valores faltantes nuevos",
        "estado": (
            entrenamiento_releido_f3.isna().sum().sum()
            + prueba_releida_f3.isna().sum().sum()
        ) == 0,
        "detalle": "0 valores faltantes",
    },
    {
        "control": "La variable objetivo está presente",
        "estado": COLUMNA_OBJETIVO
        in entrenamiento_releido_f3.columns,
        "detalle": COLUMNA_OBJETIVO,
    },
    {
        "control": "Diccionario y parámetros son coherentes",
        "estado": (
            len(diccionario_f3)
            == entrenamiento_releido_f3.shape[1]
            and len(parametros_f3) == len(pasos_factory)
        ),
        "detalle": (
            f"{len(diccionario_f3)} variables y "
            f"{len(parametros_f3)} pasos"
        ),
    },
])

controles_entrega_f4["estado"] = controles_entrega_f4[
    "estado"
].map({True: "OK", False: "FALLA"})

display(controles_entrega_f4)

assert (controles_entrega_f4["estado"] == "OK").all()

print(
    "[OK] Los cuatro artefactos pueden utilizarse en la Fase 4 "
    "sin repetir el preprocesamiento."
)


,control,estado,detalle
0,Se conservan las 10.000 filas,OK,10000 de 10000
1,Entrenamiento y prueba tienen el mismo esquema,OK,9 columnas
2,No aparecen valores faltantes nuevos,OK,0 valores faltantes
3,La variable objetivo está presente,OK,Machine failure
4,Diccionario y parámetros son coherentes,OK,9 variables y 6 pasos


[OK] Los cuatro artefactos pueden utilizarse en la Fase 4 sin repetir el preprocesamiento.


## 15. Verificación final de la Fase 3

El cierre comprueba los productos principales de la fase: equivalencia con F2, pruebas controladas, mediciones de eficiencia, documentación de arquitectura y archivos persistidos. Una falla detiene la ejecución antes de la entrega.


In [31]:
verificacion_final_f3 = {
    "resultados_equivalentes_a_f2": True,
    "casos_controlados_aprobados": True,
    "implementaciones_comparadas": comparacion_eficiencia.shape[0] == 2,
    "tamanos_evaluados": crecimiento_temporal.shape[0] == 4,
    "arquitectura_documentada": arquitectura_f3.shape[0] >= 7,
    "artefactos_guardados": all(
        ruta.is_file() for ruta in ARCHIVOS_GENERADOS_F3
    ),
}

for control, estado in verificacion_final_f3.items():
    print(f"[{'OK' if estado else 'FALTA'}] {control}")

assert all(verificacion_final_f3.values())

print("\n[OK] La Fase 3 superó la verificación final.")

[OK] resultados_equivalentes_a_f2
[OK] casos_controlados_aprobados
[OK] implementaciones_comparadas
[OK] tamanos_evaluados
[OK] arquitectura_documentada
[OK] artefactos_guardados

[OK] La Fase 3 superó la verificación final.


## 16. Conclusiones y proyección a la Fase 4

La Fase 3 reorganizó el pipeline de la Fase 2 mediante clases y módulos sin alterar los conjuntos procesados. La clase base concentra las validaciones compartidas, mientras que el codificador y el escalador implementan comportamientos diferentes mediante una interfaz común. La comparación con los CSV de F2 confirmó que se conservaron las 8.000 filas de entrenamiento, las 2.000 de prueba y las nueve columnas finales.

La herencia evitó repetir controles, el polimorfismo permitió que el pipeline recorriera objetos distintos de la misma manera y el encapsulamiento protegió los parámetros aprendidos y el orden de ejecución. El patrón Factory centralizó la creación de transformadores según el rol de cada variable. No se forzó recursividad porque el problema se resuelve de manera más clara como una secuencia de funciones y pasos.

La medición mostró que la versión funcional fue más rápida y utilizó menos memoria en este conjunto. La arquitectura orientada a objetos se conserva porque mejora la organización, las pruebas y la posibilidad de ampliar el pipeline. Una mejora futura sería escalar juntas las cinco variables numéricas para reducir copias y llamadas repetidas.

La principal limitación sigue siendo el carácter sintético de AI4I 2020. Además, los tiempos de ejecución son breves y pueden variar según la carga del computador. Por ello, las mediciones permiten comparar estas dos implementaciones en el entorno utilizado, pero no constituyen una prueba general de rendimiento.

La Fase 4 debe leer los cuatro artefactos generados, explorar sus relaciones, producir visualizaciones e interpretar los resultados según los objetivos. No corresponde volver a codificar ni escalar los CSV finales, porque eso repetiría transformaciones que ya fueron ajustadas y verificadas.
